# From OSI to SLayer

**TL;DR:** SLayer can ingest an [OSI](https://open-semantic-interchange.org/) (Open Semantic Interchange) config — its `datasets`, `relationships`, and `metrics` — and turn it into queryable SLayer datasets. This notebook imports a small retail OSI config end-to-end and answers five questions, checking each answer against gold SQL.

It runs as four explicit steps (everything lands in a gitignored `.cache/` next to this notebook):

1. **Build the data** — create a tiny retail DuckDB (orders, customers, products, regions) with deterministic rows.
2. **Reference answers** — run the gold SQL *up front*, before the importer touches the file.
3. **Import** — convert `shop.osi.yaml` into SLayer datasets with `OsiToSlayerConverter`.
4. **Query** — point a SLayer client at the imported datasets and verify against gold.

> Unlike the [dbt MetricFlow demo](../11_dbt_metricflow/dbt_metricflow_nb.ipynb), this one is **fully offline** — no network access is needed at any point.

The committed OSI config ([`shop.osi.yaml`](shop.osi.yaml)) exercises every OSI feature the importer understands: `ai_context` metadata, `nonportable_extensions`, relationships that become SLayer relationships, and dataset-scoped (including a `FILTER`) and cross-dataset / multi-hop model-scoped metrics, in the current flat Ossie shape.

## Step 1 — Build the demo database

Create the retail DuckDB the OSI config binds to. OSI carries no column *types* — the importer reads them by **live introspection** of this database — so it has to exist first.

In [ ]:
import os
import sys

import pandas as pd

# The setup helper lives next to this notebook.
sys.path.insert(0, os.getcwd())

from setup_osi import (
    build_shop_duckdb,
    convert_osi_to_slayer,
    compute_gold,
    OSI_CONFIG,
    DB_PATH,
    DATASETS_DIR,
    DATASOURCE_NAME,
)
from slayer.async_utils import run_sync
from slayer.client.slayer_client import SlayerClient
from slayer.storage.yaml_storage import YAMLStorage

db_path = build_shop_duckdb(DB_PATH)
print(f"Built retail DuckDB '{db_path.name}' with orders, customers, products, regions.")

## Step 2 — Reference answers (gold SQL)

We run the gold SQL **first**, up front, and stash the expected numbers.

Why up front? The importer introspects the datasource live, opening a read-write SQLAlchemy engine on the DuckDB file — and DuckDB won't let a second raw connection share the file under a different configuration. So every gold query runs *before* the import (Step 3) touches the file.

In [ ]:
# The reference answers live in setup_osi.compute_gold — shared with the
# agent-workflow notebook so both demos check against identical numbers.
GOLD = compute_gold(DB_PATH)

print("gold total amount :", GOLD["total"])
print("gold aov          :", GOLD["aov"])
print("gold cust_reach   :", GOLD["cust_reach"])
print("gold rev_plus_pop :", GOLD["rev_plus_pop"])
pd.DataFrame(GOLD["by_region"])

## Step 3 — Import the OSI config

This is the heart of the demo. `convert_osi_to_slayer` registers the DuckDB as a SLayer datasource, parses `shop.osi.yaml`, and runs [`OsiToSlayerConverter`](../../osi/osi_import.md):

- each OSI **dataset** becomes a SLayer dataset (column *types* from live introspection),
- each **relationship** becomes a relationship (a LEFT join),
- each **metric** folds into a `DatasetMetric` expression on the dataset the converter picks as its anchor.

The converter reports anything it can't express cleanly — here, everything converts.

In [ ]:
result = convert_osi_to_slayer(OSI_CONFIG, DATASETS_DIR, DB_PATH)

print("Imported datasets:", [d.name for d in result.datasets])
unconverted, dropped = result.tally()
print(f"{unconverted} metrics unconverted, {dropped} items dropped")
print()
print(result.render_report())

## Step 4 — The imported metadata survived

OSI's `ai_context` and `nonportable_extensions` don't just decorate the source file — the importer carries them onto the SLayer entities so agents can read them. Below, the `orders.amount` field keeps its label, its `ai_context` instructions as a description, and both blocks in `meta`; the `orders -> customers` relationship keeps its OSI relationship `ai_context` as a description.

In [ ]:
storage = YAMLStorage(base_dir=str(DATASETS_DIR))
orders = run_sync(storage.get_dataset("orders", data_source=DATASOURCE_NAME))

amount = next(f for f in orders.fields if f.name == "amount")
print("orders.amount.label       :", amount.label)
print("orders.amount.description :", amount.description)
print("orders.amount.meta        :", amount.meta)
print()
for r in orders.relationships:
    print(f"relationship orders -> {r.to}: {r.description!r}")
print()
print("metrics on orders:", [m.name for m in orders.metrics])

## Query the imported datasets

Point a SLayer client at the freshly imported datasets. Every query below is plain SLayer JSON — the OSI origin is now invisible.

In [ ]:
client = SlayerClient(storage=YAMLStorage(base_dir=str(DATASETS_DIR)))


def approx(a, b, tol=1e-9):
    return abs(a - b) < tol


def show(got, exp):
    """Print a SLayer scalar answer next to its gold value."""
    print("SLayer:", got, " gold:", exp)

### Query 1 — total order value

A simple dataset-scoped metric: `total_amount` is `SUM(amount)` on the `orders` dataset, imported as the SLayer expression `sum(amount)`.

In [ ]:
q1 = {"population": "orders", "metrics": ["orders.total_amount"]}
rows = client.query_sync(q1).data
print("SLayer:", rows)

assert approx(rows[0]["orders.total_amount"], GOLD["total"])
print(f"OK — SLayer and gold agree: {GOLD['total']}")

### Query 2 — total by region (multi-hop relationship path)

`orders.total_amount` grouped by `customers.regions.name` walks two relationships the converter carried over from OSI — `orders -> customers -> regions` — with no SQL written by hand.

In [ ]:
q2 = {
    "population": "orders",
    "metrics": ["orders.total_amount"],
    "dimensions": ["customers.regions.name"],
    "order": [{"field": "customers.regions.name"}],
}
rows = client.query_sync(q2).data
display(pd.DataFrame(rows))

slayer_pairs = [(r["customers.regions.name"], r["orders.total_amount"]) for r in rows]
gold_pairs = [(g["region"], g["amount"]) for g in GOLD["by_region"]]
assert slayer_pairs == gold_pairs, f"{slayer_pairs} != {gold_pairs}"
print(f"OK — SLayer and gold agree on {len(slayer_pairs)} region(s)")

### Query 3 — average order value (derived metric)

`aov` came from the OSI expression `SUM(amount) / COUNT(*)` — a **derived** metric over two aggregates, imported as the expression `sum(amount) / count(*)`.

In [ ]:
q3 = {"population": "orders", "metrics": ["orders.aov"]}
rows = client.query_sync(q3).data
show(rows[0]["orders.aov"], GOLD["aov"])

assert approx(rows[0]["orders.aov"], GOLD["aov"])
print("OK — average order value matches gold")

### Query 4 — amount per distinct customer (cross-dataset metric)

`cust_reach` = `SUM(orders.amount) / COUNT(DISTINCT customers.customer_id)`. The metric spans two datasets; the converter anchors it on `orders`, the dataset that reaches `customers` without crossing a relationship in its fan-out direction, so it imports as `sum(amount) / count_distinct(customers.customer_id)`.

In [ ]:
q4 = {"population": "orders", "metrics": ["orders.cust_reach"]}
rows = client.query_sync(q4).data
show(rows[0]["orders.cust_reach"], GOLD["cust_reach"])

assert approx(rows[0]["orders.cust_reach"], GOLD["cust_reach"])
print("OK — amount per distinct customer matches gold")

### Query 5 — amount plus region population (multi-hop metric)

`rev_plus_pop` = `SUM(orders.amount) + SUM(regions.population)` reaches `regions` two relationships away (`orders -> customers -> regions`).

The subtle part: SLayer aggregates `regions.population` **at the regions grain** (each distinct region counted once), so adding this metric can't multiply the order rows. A naive five-way join would instead add a region's population *once per order* in that region and badly over-count. Our gold mirrors SLayer's isolation — summing population over the distinct regions the orders reach — and the two agree.

In [ ]:
q5 = {"population": "orders", "metrics": ["orders.rev_plus_pop"]}
rows = client.query_sync(q5).data
show(rows[0]["orders.rev_plus_pop"], GOLD["rev_plus_pop"])

assert approx(rows[0]["orders.rev_plus_pop"], GOLD["rev_plus_pop"])
print("OK — SLayer isolates the joined aggregate; both agree at", GOLD["rev_plus_pop"])

## Recap

Starting from an OSI config we did not write, SLayer:

- imported each `dataset` as a queryable SLayer dataset, with real column types from live introspection,
- turned OSI `relationships` into SLayer relationships, so metrics and dimensions reach across them (including multi-hop),
- folded simple, derived, cross-dataset, and multi-hop `metrics` into `DatasetMetric` expressions,
- and preserved `ai_context` / `nonportable_extensions` as descriptions and `meta` for agents to read.

Every answer matched gold SQL — including the multi-hop metric, where SLayer's sub-query isolation kept the added aggregate from fanning out the order rows.

### Further reading

- [Importing OSI configs](../../osi/osi_import.md) — the full conversion reference, including exactly what converts and what fails cleanly.
- The one-liner this notebook mirrors: `slayer import-osi shop.osi.yaml --datasource shop_osi` (against a registered datasource).